# Ablation and Analysis
- This is the third notebook for the project: AI-Based Resume Screening and Hiring Prediction System
## Purpose:
We will compare model performance using different feature groups:
- Numeric-only model
- Text-only model
- Combined model

We also included:
- Performance comparison table

## Discussion:
- Best performing model
- Effect of different feature groups
- Limitations of the current approach
- Possible improvements

## Setup

In [12]:
# Import

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve

In [13]:
# Load the data
import os
import pandas as pd

DATA_DIR = "data/processed"

# Load feature matrices
X_train = pd.read_parquet(os.path.join(DATA_DIR, "X_train.parquet"))
X_test = pd.read_parquet(os.path.join(DATA_DIR, "X_test.parquet"))

# Load labels
y_train = pd.read_parquet(os.path.join(DATA_DIR, "y_train.parquet"))["target"]
y_test = pd.read_parquet(os.path.join(DATA_DIR, "y_test.parquet"))["target"]

## Ablation table:  Numeric vs Text vs Combined

In [15]:
# Encode labels (Hire / Reject)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Separate feature groups
text_cols = [c for c in X_train.columns if "tfidf" in c.lower()]
numeric_cols = [c for c in X_train.columns if c not in text_cols]

X_train_num = X_train[numeric_cols]
X_test_num = X_test[numeric_cols]

X_train_text = X_train[text_cols]
X_test_text = X_test[text_cols]

X_train_combined = X_train
X_test_combined = X_test

# Train XGBoost
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

# Evaluation Function
def evaluate(Xtr, Xte, ytr, yte):

    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss"
    )

    model.fit(Xtr, ytr)
    pred = model.predict(Xte)

    return {
        "Accuracy": accuracy_score(yte, pred),
        "Precision": precision_score(yte, pred),
        "Recall": recall_score(yte, pred),
        "F1": f1_score(yte, pred)
    }

# Report
results = []

results.append(("Numeric", evaluate(X_train_num, X_test_num, y_train, y_test)))
results.append(("Text", evaluate(X_train_text, X_test_text, y_train, y_test)))
results.append(("Combined", evaluate(X_train_combined, X_test_combined, y_train, y_test)))

results_df = pd.DataFrame(
    [{**r[1], "Feature Group": r[0]} for r in results]
)

results_df

,Accuracy,Precision,Recall,F1,Feature Group
0,0.975,0.987578,0.981481,0.984520,Numeric
1,0.735,0.804469,0.888889,0.844575,Text
2,0.965,0.981366,0.975309,0.978328,Combined


## Discussion:
- The numeric-only model had the highest F1-score, indicating that numeric features are the most important predictors in the model.
- The text features provided meaningful information, but may introduce noise.
- The limiation of the current is we used TF-IDF to represent the text features which introduce noise becasue of high dimensionality.